## 测试：`fit-4hdnnp-NaCl-initial` 中 `Q_tot` 是否会被正确读取

目的：验证在前向中 `charge -> system_charge -> ChargeEq` 的链路是否正常。

你将看到三组总电荷（按结构/graph）：
- `Q_tot_ref`: 数据集提供的原子电荷求和
- `Q_tot_model_system_charge`: 模型前向里 `SystemChargeFromAtomicCharges` 写入的 `system_charge`
- `Q_tot_from_qeq`: 模型预测的 `q_eq` 按结构求和

期望：三者应一致（允许数值误差）。

> 注意：如果你刚改过 `cace/representations/cace_representation.py`，请先重启 kernel 再运行，避免旧代码缓存。

In [1]:
import os
import sys

# Jupyter 里没有 __file__，用当前工作目录作为基准。
# 运行时建议先把工作目录切到本 notebook 所在目录（Cursor 通常会自动做到）。
HERE = os.getcwd()
ROOT_DIR = os.path.abspath(os.path.join(HERE, ".."))  # 指向 CACE-SOG-Qeq
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

os.environ["PYTHONWARNINGS"] = "ignore"  # 全局忽略所有 warnings

print("HERE:", HERE)
print("ROOT_DIR (added to sys.path):", ROOT_DIR)

HERE: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl-initial
ROOT_DIR (added to sys.path): /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq


In [2]:
import os
import numpy as np
import torch
import torch.nn as nn

import cace
from cace.representations import Cace
from cace.modules import PolynomialCutoff
from cace.modules import BesselRBF
from cace.tools.scatter import scatter_sum
from cace.tools import torch_geometric

from cace.models.atomistic import NeuralNetworkPotential
from cace.data.extxyz_charge import get_dataset_from_extxyz_with_charge

torch.set_default_dtype(torch.float32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
train_path = os.path.join(HERE, "NaCl.xyz")
ckpt_path = os.path.join(HERE, "checkpoint.pt")
print("train_path:", train_path)
print("ckpt_path:", ckpt_path)
print("ckpt exists:", os.path.exists(ckpt_path))

device: cpu
train_path: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl-initial/NaCl.xyz
ckpt_path: /work/home/acrb3qk4vo/SOG-Qeq/SOG-Net/CACE-SOG-Qeq/fit-4hdnnp-NaCl-initial/checkpoint.pt
ckpt exists: False


In [3]:
cutoff = 5.29
Fourier_node = 18
batch_size = 5

collection = get_dataset_from_extxyz_with_charge(
    train_path=train_path,
    cutoff=cutoff,
    valid_fraction=0.1,
    seed=1,
    atomic_energies={11: -4417.07609365649, 17: -12516.880649933015},
)

train_loader = torch_geometric.DataLoader(
    dataset=collection.train,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)
valid_loader = torch_geometric.DataLoader(
    dataset=collection.valid,
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
)

print("num_train_graphs:", len(collection.train))
print("num_valid_graphs:", len(collection.valid))

num_train_graphs: 4500
num_valid_graphs: 500


In [4]:
class SystemChargeFromAtomicCharges(nn.Module):
    def __init__(self, charges_key: str = "charge", output_key: str = "system_charge"):
        super().__init__()
        self.charges_key = charges_key
        self.output_key = output_key
        self.model_outputs = [output_key]

    def forward(self, data: dict, **kwargs):
        if self.charges_key not in data or data[self.charges_key] is None:
            if data.get("batch", None) is None:
                num_graphs = 1
            else:
                num_graphs = int(data["batch"].max().item()) + 1 if data["batch"].numel() > 0 else 1
            data[self.output_key] = torch.zeros(
                (num_graphs,), device=data["positions"].device, dtype=data["positions"].dtype
            )
            return data

        q = data[self.charges_key]
        if q.dim() > 1:
            q = q.view(-1)
        if data.get("batch", None) is None:
            system_q = q.sum().view(1)
        else:
            system_q = scatter_sum(q, data["batch"], dim=0)
        data[self.output_key] = system_q
        return data


def build_model() -> NeuralNetworkPotential:
    """只构建与 Q_tot 链路相关的前向（不包含 Forces）。

    说明：`Forces` 会触发对 `positions` 的 autograd 求导；为了让本测试可以在
    `torch.no_grad()` 下运行，我们这里不把 `Forces` 放进 `output_modules`。
    """

    radial_basis = BesselRBF(cutoff=cutoff, n_rbf=6, trainable=True)
    cutoff_fn = PolynomialCutoff(cutoff=cutoff)

    rep = Cace(
        zs=[11, 17],
        n_atom_basis=2,
        embed_receiver_nodes=True,
        cutoff=cutoff,
        cutoff_fn=cutoff_fn,
        radial_basis=radial_basis,
        n_radial_basis=8,
        max_l=3,
        max_nu=3,
        num_message_passing=0,
        type_message_passing=["Bchi"],
        args_message_passing={"Bchi": {"shared_channels": False, "shared_l": False}},
        device=device,
        timeit=False,
        # 这里即使只写 atomic_numbers，也应当能工作：我们已在 Cace.forward 中强制透传 charge
        forward_features=["atomic_numbers"],
    ).to(device)

    sr_energy = cace.modules.atomwise.Atomwise(
        n_layers=3,
        output_key="SR_energy",
        n_hidden=[32, 16],
        use_batchnorm=False,
        add_linear_nn=True,
    )

    chi = cace.modules.Atomwise(
        n_layers=3,
        n_hidden=[24, 12],
        n_out=1,
        per_atom_output_key="chi",
        output_key="tot_chi",
        residual=False,
        add_linear_nn=True,
        post_process=torch.square,
        bias=False,
    )

    system_charge_from_q = SystemChargeFromAtomicCharges(charges_key="charge", output_key="system_charge")

    charge_eq = cace.modules.ChargeEq(
        dl=1.5,
        sigma=1.0,
        elements=[11, 17],
        feature_key="chi",
        output_key="q_eq",
        ewald_key="SOG_potential",
        system_charge=None,
        remove_self_interaction=True,
        aggregation_mode="sum",
        use_sog_kernel=True,
        sog_num_components=Fourier_node,
    )

    e_add = cace.modules.FeatureAdd(
        feature_keys=["SR_energy", "SOG_potential"],
        output_key="CACE_energy",
    )

    model = NeuralNetworkPotential(
        input_modules=None,
        representation=rep,
        output_modules=[sr_energy, chi, system_charge_from_q, charge_eq, e_add],
    ).to(device)

    # 与 initial 脚本一致：BSA 初始化（即使只做推理也无害）
    charge_eq.init_sog_from_bsa(r_cut=cutoff, b=2.0)

    return model


model = build_model()
model.eval()
print(model)

NeuralNetworkPotential(
  (postprocessors): ModuleList()
  (representation): Cace(
    (node_onehot): NodeEncoder(num_classes=2)
    (node_embedding_sender): NodeEmbedding(num_classes=2, embedding_dim=2)
    (node_embedding_receiver): NodeEmbedding(num_classes=2, embedding_dim=2)
    (edge_coding): EdgeEncoder(directed=True)
    (radial_basis): BesselRBF(cutoff=5.289999961853027, n_rbf=6, trainable=True)
    (cutoff_fn): PolynomialCutoff(p=6.0, cutoff=5.289999961853027)
    (angular_basis): AngularComponent(l_max=3)
    (radial_transform): SharedRadialLinearTransform(
      (weights): ParameterList(
          (0): Parameter containing: [torch.float32 of size 6x8x4]
          (1): Parameter containing: [torch.float32 of size 6x8x4]
          (2): Parameter containing: [torch.float32 of size 6x8x4]
          (3): Parameter containing: [torch.float32 of size 6x8x4]
      )
    )
    (symmetrizer): Symmetrizer()
    (message_passing_list): ModuleList()
  )
  (input_modules): ModuleList(
  

In [5]:
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
    print("loaded checkpoint.pt")
    print("missing keys:", len(missing))
    print("unexpected keys:", len(unexpected))
else:
    print("No checkpoint.pt found; will test with randomly initialized weights.")

No checkpoint.pt found; will test with randomly initialized weights.


In [9]:
def compute_qtot_per_graph(q_atom_1d: torch.Tensor, batch_index: torch.Tensor) -> torch.Tensor:
    q_atom_1d = q_atom_1d.view(-1)
    return scatter_sum(q_atom_1d, batch_index, dim=0)


batch = next(iter(valid_loader))
batch = batch.to(device)
batch_dict = batch.to_dict()

with torch.no_grad():
    pred = model(batch_dict, training=False)

q_ref = batch_dict["charge"].view(-1)
batch_index = batch_dict["batch"].view(-1)

Q_tot_ref = compute_qtot_per_graph(q_ref, batch_index)

Q_tot_model_system_charge = pred.get("system_charge", None)
if Q_tot_model_system_charge is not None:
    Q_tot_model_system_charge = Q_tot_model_system_charge.view(-1)

q_qeq = pred.get("q_eq", None)
Q_tot_from_qeq = None
if q_qeq is not None:
    Q_tot_from_qeq = compute_qtot_per_graph(q_qeq.view(-1), batch_index)

norm_factor = None
try:
    # ChargeEq 内部会用 system_Q / normalization_factor 作为约束右端
    for m in model.output_modules:
        if m.__class__.__name__ == "ChargeEq":
            norm_factor = float(m.normalization_factor)
            break
except Exception:
    norm_factor = None

print("ChargeEq normalization_factor:", norm_factor)

print("Q_tot_ref:", Q_tot_ref.detach().cpu().numpy())
print(
    "Q_tot_model_system_charge:",
    None if Q_tot_model_system_charge is None else Q_tot_model_system_charge.detach().cpu().numpy(),
)
print("Q_tot_from_qeq (raw sum):", None if Q_tot_from_qeq is None else Q_tot_from_qeq.detach().cpu().numpy())

Q_tot_from_qeq_physical = None
if Q_tot_from_qeq is not None and norm_factor is not None:
    Q_tot_from_qeq_physical = Q_tot_from_qeq * norm_factor
    print("Q_tot_from_qeq (rescaled):", Q_tot_from_qeq_physical.detach().cpu().numpy())

if Q_tot_model_system_charge is not None:
    print("max|system_charge - ref|:", (Q_tot_model_system_charge - Q_tot_ref).abs().max().item())

if Q_tot_from_qeq_physical is not None:
    print("max|rescaled sum(q_eq) - ref|:", (Q_tot_from_qeq_physical - Q_tot_ref).abs().max().item())
elif Q_tot_from_qeq is not None:
    print("max|raw sum(q_eq) - ref|:", (Q_tot_from_qeq - Q_tot_ref).abs().max().item())

ChargeEq normalization_factor: 0.10538150852788286
Q_tot_ref: [1.         1.         0.9999999  0.9999999  0.99999994]
Q_tot_model_system_charge: [1.         1.         0.9999999  0.9999999  0.99999994]
Q_tot_from_qeq (raw sum): [9.489327 9.489333 9.489331 9.489329 9.489329]
Q_tot_from_qeq (rescaled): [0.99999964 1.0000002  1.0000001  0.9999999  0.9999999 ]
max|system_charge - ref|: 0.0
max|rescaled sum(q_eq) - ref|: 3.5762786865234375e-07


In [10]:
# 可选：多拿几个 batch 看统计

def eval_many(n_batches: int = 10):
    Qref_all = []
    Qsys_all = []
    Qqeq_all = []

    it = iter(valid_loader)
    for _ in range(n_batches):
        try:
            b = next(it)
        except StopIteration:
            break
        b = b.to(device)
        bd = b.to_dict()
        with torch.no_grad():
            pr = model(bd, training=False)

        batch_index = bd["batch"].view(-1)
        Qref = compute_qtot_per_graph(bd["charge"].view(-1), batch_index).detach().cpu().numpy()
        Qref_all.append(Qref)

        if "system_charge" in pr and pr["system_charge"] is not None:
            Qsys_all.append(pr["system_charge"].view(-1).detach().cpu().numpy())

        if "q_eq" in pr and pr["q_eq"] is not None:
            Qqeq_all.append(compute_qtot_per_graph(pr["q_eq"].view(-1), batch_index).detach().cpu().numpy())

    Qref_all = np.concatenate(Qref_all) if len(Qref_all) else np.array([])
    Qsys_all = np.concatenate(Qsys_all) if len(Qsys_all) else np.array([])
    Qqeq_all = np.concatenate(Qqeq_all) if len(Qqeq_all) else np.array([])

    print("Q_tot_ref mean/std:", Qref_all.mean(), Qref_all.std())
    if Qsys_all.size:
        print("system_charge mean/std:", Qsys_all.mean(), Qsys_all.std())
        print("MAE(system_charge-ref):", np.mean(np.abs(Qsys_all - Qref_all)))
    if Qqeq_all.size:
        print("sum(q_eq) mean/std:", Qqeq_all.mean(), Qqeq_all.std())
        if norm_factor is not None:
            Qqeq_phys = Qqeq_all * norm_factor
            print("rescaled sum(q_eq) mean/std:", Qqeq_phys.mean(), Qqeq_phys.std())
            print("MAE(rescaled sum(q_eq)-ref):", np.mean(np.abs(Qqeq_phys - Qref_all)))
        else:
            print("MAE(sum(q_eq)-ref):", np.mean(np.abs(Qqeq_all - Qref_all)))


eval_many(10)

Q_tot_ref mean/std: 1.0 8.172577e-08
system_charge mean/std: 1.0 8.172577e-08
MAE(system_charge-ref): 0.0
sum(q_eq) mean/std: 9.48933 1.5018478e-06
rescaled sum(q_eq) mean/std: 1.0 1.760104e-07
MAE(rescaled sum(q_eq)-ref): 1.14440915e-07
